# Stage 11: Reportable Confidence Intervals

This notebook runs the bootstrap confidence-interval utility on the final reportable counting artifacts.

Primary targets:
- `squat_tcn_l1_channels96`
- `pose_count_tcn_pull_up_seq192`
- `rgb_count_tcn_push_up_seq128`
- optional routed Stage 8 output

The goal is to supplement point estimates with bootstrap intervals for `MAE`, `RMSE`, and `Within-1` on the validation split.

In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')
LOCAL_PROJECT_ROOT = Path('/content/CV_Image_pose_detection')
BOOTSTRAP_REL = Path('artifacts/3_Modeling/bootstrap_count_confidence_intervals.py')

if LOCAL_PROJECT_ROOT.exists():
    src = LOCAL_PROJECT_ROOT / BOOTSTRAP_REL
    dst = DRIVE_PROJECT_ROOT / BOOTSTRAP_REL
    dst.parent.mkdir(parents=True, exist_ok=True)
    if src.exists():
        shutil.copy2(src, dst)
        print(f'Synced {src} -> {dst}')

TRAINING_OUTPUTS = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs'
BOOTSTRAP_SCRIPT = DRIVE_PROJECT_ROOT / BOOTSTRAP_REL
TRAINING_OUTPUTS.mkdir(parents=True, exist_ok=True)
print('Using project root:', DRIVE_PROJECT_ROOT)
print('Using bootstrap script:', BOOTSTRAP_SCRIPT)
print('TRAINING_OUTPUTS exists:', TRAINING_OUTPUTS.exists())
if TRAINING_OUTPUTS.exists():
    run_dirs = sorted(p.name for p in TRAINING_OUTPUTS.iterdir() if p.is_dir())
    print('Available training_outputs dirs (first 30):', run_dirs[:30])


In [ ]:
def resolve_predictions(run_name: str) -> Path:
    direct = TRAINING_OUTPUTS / run_name / 'predictions.csv'
    if direct.exists():
        return direct
    matches = sorted(TRAINING_OUTPUTS.glob(f'**/{run_name}/predictions.csv'))
    if matches:
        return matches[0]
    return direct

RUNS = [
    {
        'label': 'squat_pose_best',
        'run_name': 'squat_tcn_l1_channels96',
        'predictions_csv': resolve_predictions('squat_tcn_l1_channels96'),
        'exercise': 'squat',
    },
    {
        'label': 'pull_up_pose_route',
        'run_name': 'pose_count_tcn_pull_up_seq192',
        'predictions_csv': resolve_predictions('pose_count_tcn_pull_up_seq192'),
        'exercise': 'pull_up',
    },
    {
        'label': 'push_up_rgb_route',
        'run_name': 'rgb_count_tcn_push_up_seq128',
        'predictions_csv': resolve_predictions('rgb_count_tcn_push_up_seq128'),
        'exercise': 'push_up',
    },
]

ROUTED_CANDIDATES = [
    TRAINING_OUTPUTS / 'routed_exercise_dependent_counting' / 'routed_predictions.csv',
    TRAINING_OUTPUTS / 'routed_stage8' / 'routed_predictions.csv',
]
for routed_csv in ROUTED_CANDIDATES:
    if routed_csv.exists():
        RUNS.append({
            'label': 'stage8_routed',
            'run_name': routed_csv.parent.name,
            'predictions_csv': routed_csv,
            'exercise': None,
        })
        break

BOOTSTRAP_SAMPLES = 5000
SEED = 7
CONFIDENCE_LEVEL = 0.95

for run in RUNS:
    print(run['label'], run['run_name'], run['predictions_csv'], run['exercise'], 'exists=', run['predictions_csv'].exists())


In [ ]:
import subprocess

bootstrap_failures = []
bootstrap_outputs = []

for run in RUNS:
    pred_csv = run['predictions_csv']
    if not pred_csv.exists():
        bootstrap_failures.append({
            'label': run['label'],
            'run_name': run.get('run_name'),
            'reason': f'missing predictions: {pred_csv}',
        })
        continue

    output_json = pred_csv.with_name('bootstrap_confidence_intervals.json')
    cmd = [
        'python', '-u', str(BOOTSTRAP_SCRIPT),
        '--predictions-csv', str(pred_csv),
        '--split', 'valid',
        '--bootstrap-samples', str(BOOTSTRAP_SAMPLES),
        '--confidence-level', str(CONFIDENCE_LEVEL),
        '--seed', str(SEED),
        '--output-json', str(output_json),
    ]
    if run['exercise']:
        cmd.extend(['--exercise', run['exercise']])

    print('\nRunning:', ' '.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print('[stderr]')
        print(result.stderr)
    if result.returncode != 0:
        bootstrap_failures.append({
            'label': run['label'],
            'run_name': run.get('run_name'),
            'reason': f'returncode={result.returncode}',
        })
        continue

    bootstrap_outputs.append({
        'label': run['label'],
        'exercise': run['exercise'],
        'output_json': output_json,
    })

print('Completed:', len(bootstrap_outputs))
if bootstrap_failures:
    print('Failures:')
    for failure in bootstrap_failures:
        print(' -', failure)


In [ ]:
import json
import pandas as pd

rows = []
for item in bootstrap_outputs:
    summary = json.loads(item['output_json'].read_text(encoding='utf-8'))
    metrics = summary['metrics']
    rows.append({
        'label': item['label'],
        'exercise': item['exercise'] or 'all',
        'rows': summary['rows'],
        'mae': metrics['mae']['point_estimate'],
        'mae_ci_low': metrics['mae']['ci_low'],
        'mae_ci_high': metrics['mae']['ci_high'],
        'rmse': metrics['rmse']['point_estimate'],
        'rmse_ci_low': metrics['rmse']['ci_low'],
        'rmse_ci_high': metrics['rmse']['ci_high'],
        'within_1': metrics['within_1']['point_estimate'],
        'within_1_ci_low': metrics['within_1']['ci_low'],
        'within_1_ci_high': metrics['within_1']['ci_high'],
    })

ci_df = pd.DataFrame(rows)
if ci_df.empty:
    print('No bootstrap summaries found yet.')
else:
    display(ci_df)


Interpretation notes:
- Use these intervals beside the point estimates in the report.
- On very small validation surfaces, expect wide intervals and avoid over-claiming small metric differences.
- If the routed Stage 8 predictions are not present locally in Drive, that row will be skipped automatically.